# thư viện sử dụng

In [47]:
import numpy as np
import pandas as pd
from pathlib import Path

# Đường dẫn đến dữ liệu thô

In [48]:

train_path = Path(r"..\..\data\raw\train.csv")


# Đọc dữ liệu

In [49]:
df = pd.read_csv(train_path)

In [50]:
print("Kích thước dữ liệu ban đầu:", df.shape)
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Số lượng missing values theo cột:")
print(missing)

Kích thước dữ liệu ban đầu: (1460, 81)
Số lượng missing values theo cột:
LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64


# 1. Điền các giá trị 0 (diện tích/số lượng = 0 khi không có tiện ích)

In [51]:
ZERO_FILL_COLS = [
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
]

for col in ZERO_FILL_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# 2. Điền các giá trị "None" (ngụ ý không có tiện ích tương ứng)

In [52]:
NA_SEMANTIC_MAP = {
    "PoolQC": "No Pool",
    "Alley": "No alley access",
    "Fence": "No Fence",
    "FireplaceQu": "No Fireplace",
    "GarageType": "No Garage",
    "GarageFinish": "No Garage",
    "GarageQual": "No Garage",
    "GarageCond": "No Garage",
    "BsmtQual": "No Basement",
    "BsmtCond": "No Basement",
    "BsmtExposure": "No Basement",
    "BsmtFinType1": "No Basement",
    "BsmtFinType2": "No Basement",
    "MiscFeature": "No Misc Feature",
}

for col, fill_value in NA_SEMANTIC_MAP.items():
    if col in df.columns:
        df[col] = df[col].fillna(fill_value)

# 3. Điền median cho các cột số thiếu thật sự

In [53]:
MEDIAN_FILL_COLS = [
    "LotFrontage",
]

for col in MEDIAN_FILL_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())


# 4. Điền mode cho các cột phân loại thiếu thật sự

In [54]:
MODE_FILL_COLS = [
    "MSZoning",
    "Utilities",
    "Exterior1st",
    "Exterior2nd",
    "Electrical",
    "KitchenQual",
    "Functional",
    "SaleType",
]

for col in MODE_FILL_COLS:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)

# 5. Xử lý ngoại lai (outliers)

## Ngoại lai: diện tích sống cực lớn nhưng giá bán bất thường rẻ

In [55]:
outlier_mask = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)
outlier_indices = df.index[outlier_mask]
print("Số lượng ngoại lai phát hiện (GrLivArea > 4000 & SalePrice < 300000):", len(outlier_indices))
print("Chỉ số ngoại lai:", outlier_indices.tolist())

Số lượng ngoại lai phát hiện (GrLivArea > 4000 & SalePrice < 300000): 2
Chỉ số ngoại lai: [523, 1298]


In [56]:
df = df.drop(index=outlier_indices).reset_index(drop=True)
print("Kích thước dữ liệu sau khi làm sạch:", df.shape)
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print("Số lượng missing values còn lại:")
print(remaining_missing)


Kích thước dữ liệu sau khi làm sạch: (1458, 81)
Số lượng missing values còn lại:
MasVnrType     872
MasVnrArea       8
GarageYrBlt     81
dtype: int64


In [57]:
print(df.groupby("MasVnrType", dropna=False)["MasVnrArea"].describe())

            count        mean         std   min    25%    50%    75%     max
MasVnrType                                                                  
BrkCmn       15.0  247.666667  182.423239  41.0   91.5  192.0  381.5   621.0
BrkFace     445.0  259.008989  215.137888   0.0  114.0  202.0  336.0  1600.0
Stone       126.0  230.738095  159.259546   0.0  110.5  202.5  292.5   860.0
NaN         864.0    1.094907   18.569659   0.0    0.0    0.0    0.0   344.0


In [58]:
print(df.loc[
    df["GarageYrBlt"].isna(),
    ["GarageType", "GarageCars", "GarageArea"]
].head(10))

    GarageType  GarageCars  GarageArea
39   No Garage           0           0
48   No Garage           0           0
78   No Garage           0           0
88   No Garage           0           0
89   No Garage           0           0
99   No Garage           0           0
108  No Garage           0           0
125  No Garage           0           0
127  No Garage           0           0
140  No Garage           0           0


# xử lý tiếp các dữ liệu đặc biệt

In [59]:
mask_none = df["MasVnrArea"].eq(0)

df.loc[mask_none, "MasVnrType"] = (
    df.loc[mask_none, "MasVnrType"].fillna("None")
)

In [60]:
mask_unknown = (
    df["MasVnrType"].isna()
    & df["MasVnrArea"].gt(0)
)

In [61]:
df.loc[mask_unknown, "MasVnrType"] = "Unknown"

In [62]:
mask_both = (
    df["MasVnrType"].isna()
    & df["MasVnrArea"].isna()
)
df.loc[mask_both, "MasVnrType"] = "Unknown"


In [63]:
df["MasVnrArea"] = df["MasVnrArea"].fillna(
    df["MasVnrArea"].median()
)

# Tạo feature cho biết nhà có garage hay không

In [64]:
df["HasGarage"] = (
    df["GarageCars"].fillna(0) > 0
).astype(int)


# Nhà không có garage -> mã hóa năm xây bằng 0


In [65]:
df.loc[
    df["HasGarage"].eq(0) & df["GarageYrBlt"].isna(),
    "GarageYrBlt"
] = 0

# Nhà có garage nhưng thiếu năm xây -> median

In [66]:
garage_median = df.loc[
    df["HasGarage"].eq(1),
    "GarageYrBlt"
].median()

In [67]:
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(
    garage_median
)

In [68]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print("Số lượng missing values còn lại:")
print(remaining_missing)

Số lượng missing values còn lại:
Series([], dtype: int64)


In [ ]:
df.to_excel("cleaned_data.xlsx", index=False)